In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# MERRA2 O3 column OCT-SEP evolution (debug version)
#   - 30–70 hPa
#   - 60–90N polar cap partial O3 column (DU)
#   - Plot climatology ±1σ
#   - Overlay Top5 lowest-O3 years
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

# ============================================================
# Paths
# ============================================================
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
OUT_DIR   = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots/MERRA2_O3_EVOLUTION"
os.makedirs(OUT_DIR, exist_ok=True)

TAG = "30_70hPa"
N_PREV = 91
N_DAYS_EXPECTED = 365   # 固定 noleap 年长度
EXTREME_COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

O3_NC = os.path.join(DATA_BASE, "MERRA2", f"O3_pc_MERRA2_{TAG}.nc")
LOW10_TXT = os.path.join(DATA_BASE, "MERRA2", f"lowest10_years_sorted_{TAG}.txt")

# ============================================================
# Style
# ============================================================
rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ============================================================
# Helpers
# ============================================================
def ensure_daily_unique(da):
    da = da.sortby("time")
    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)
    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError("Duplicated daily timestamps detected.")
    return da


def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)


def print_year_counts(da, title="Year counts"):
    years = np.unique(da.time.dt.year.values.astype(int))
    print(f"\n[DEBUG] {title}")
    for y in years:
        cnt = int((da.time.dt.year == y).sum().values)
        print(f"  Year {y}: {cnt} days")


def build_octsep_series(da, year, n_days=365, n_prev=91, debug=False):
    """
    Build one Oct-Sep series for target year:
      Oct-Dec from year-1 + Jan-Sep from year
    """
    cur = da.sel(time=da.time.dt.year == year)
    prev = da.sel(time=da.time.dt.year == (year - 1))

    cur_n = cur.sizes.get("time", 0)
    prev_n = prev.sizes.get("time", 0)

    if debug:
        print(f"[DEBUG] build_octsep_series year={year}")
        print(f"        prev year={year-1}, prev_n={prev_n}")
        print(f"        cur  year={year},   cur_n={cur_n}")

    if prev_n != n_days or cur_n != n_days:
        if debug:
            print(f"        -> INCOMPLETE: expected prev={n_days}, cur={n_days}")
        return None

    cur_arr = np.asarray(cur.values, dtype=float)
    prev_arr = np.asarray(prev.values, dtype=float)

    series = np.concatenate([
        prev_arr[n_days - n_prev:n_days],   # Oct-Dec of previous year
        cur_arr[0:n_days - n_prev]          # Jan-Sep of current year
    ])

    if debug:
        print(f"        -> OK, output length = {len(series)}")

    return series


# ============================================================
# Main
# ============================================================
print("[INFO] Loading MERRA2 O3 column:", O3_NC)
da = xr.load_dataarray(O3_NC)
da = ensure_daily_unique(da)

print(f"[DEBUG] Original total length = {da.sizes['time']}")
print(f"[DEBUG] Original first time = {da.time.values[0]}")
print(f"[DEBUG] Original last  time = {da.time.values[-1]}")

print_year_counts(da, title="Before drop_feb29")

da = drop_feb29(da)

print(f"\n[DEBUG] After drop_feb29 total length = {da.sizes['time']}")
print(f"[DEBUG] After drop_feb29 first time = {da.time.values[0]}")
print(f"[DEBUG] After drop_feb29 last  time = {da.time.values[-1]}")

print_year_counts(da, title="After drop_feb29")

years_all = np.unique(da.time.dt.year.values.astype(int))
n_days = N_DAYS_EXPECTED

print("\n[INFO] Available years:", years_all[0], "to", years_all[-1])
print("[INFO] Days per year (forced noleap):", n_days)

# Top5 extreme years
lowest10 = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
top5_years = lowest10[:5]

print("[INFO] lowest10 years:", lowest10)
print("[INFO] top5 years:", top5_years)

# 针对 top5 做额外调试
print("\n[DEBUG] Top5 year availability check:")
for y in top5_years:
    cur_n = int((da.time.dt.year == y).sum().values)
    prev_n = int((da.time.dt.year == (y - 1)).sum().values)
    print(f"  target year {y}: prev={y-1} has {prev_n} days, current={y} has {cur_n} days")

# ------------------------------------------------------------
# Climatology (all years mean/std by month-day, not dayofyear)
# ------------------------------------------------------------
# 这里继续用 dayofyear 也能做，但更稳妥是直接按 month-day 做 noleap 气候态
month = da.time.dt.month.values.astype(int)
day   = da.time.dt.day.values.astype(int)
mmdd  = np.array([f"{m:02d}-{d:02d}" for m, d in zip(month, day)])

da = da.assign_coords(mmdd=("time", mmdd))

clim_day_mean = da.groupby("mmdd").mean("time")
clim_day_std  = da.groupby("mmdd").std("time")

# 构造 Oct-Sep 的 mmdd 顺序
month_days = [
    (10,31), (11,30), (12,31), (1,31), (2,28), (3,31),
    (4,30), (5,31), (6,30), (7,31), (8,31), (9,30)
]
mmdd_order = []
for m, nd in month_days:
    for d in range(1, nd + 1):
        mmdd_order.append(f"{m:02d}-{d:02d}")

clim_octsep_mean = clim_day_mean.sel(mmdd=mmdd_order).values
clim_octsep_std  = clim_day_std.sel(mmdd=mmdd_order).values

# ------------------------------------------------------------
# Build Top5 series
# ------------------------------------------------------------
series_top5 = []
valid_top5_years = []

for y in top5_years:
    s = build_octsep_series(da, y, n_days=n_days, n_prev=N_PREV, debug=True)
    if s is None:
        print(f"[WARN] Skip year {y}: incomplete previous/current year data.")
        continue
    series_top5.append(s)
    valid_top5_years.append(int(y))

series_top5 = np.asarray(series_top5, dtype=float) if len(series_top5) > 0 else np.empty((0, n_days))

print(f"\n[DEBUG] valid_top5_years = {valid_top5_years}")
print(f"[DEBUG] number of valid top5 series = {len(valid_top5_years)}")

# ============================================================
# Plot
# ============================================================
fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.0), constrained_layout=True)

x_full = np.arange(n_days)
colors_spec = EXTREME_COLORS

# extreme years
for i, y in enumerate(valid_top5_years):
    ax.plot(
        x_full,
        series_top5[i],
        lw=1.6,
        color=colors_spec[i],
        label=f"Year {y:04d}"
    )

# climatology envelope
ax.fill_between(
    x_full,
    clim_octsep_mean - clim_octsep_std,
    clim_octsep_mean + clim_octsep_std,
    color="0.85",
    alpha=0.9,
    linewidth=0,
    zorder=0
)

ax.plot(
    x_full,
    clim_octsep_mean,
    ls="--",
    lw=1.8,
    color="black",
    label="MERRA2 climatology"
)

# Axes
ax.set_xlim(0, n_days)
ax.set_xticks([0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 334])
ax.set_xticklabels(
    ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "June", "July", "Aug", "Sep"]
)

ax.set_ylabel("Partial O$_3$ column, 30–70 hPa (DU)")
ax.set_title("MERRA2 polar-cap partial ozone column (60–90°N, 30–70 hPa)")
ax.grid(False)

# Legend
patch_clim = Patch(facecolor="0.85", edgecolor="none", label="MERRA2 ±1σ")
line_clim  = Line2D([0], [0], color="black", lw=1.8, ls="--", label="MERRA2 climatology")

handles_years = [
    Line2D([0], [0], color=colors_spec[i], lw=1.6, label=f"Year {y:04d}")
    for i, y in enumerate(valid_top5_years)
]

handles = handles_years + [line_clim, patch_clim]
ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

# Save
out_png = os.path.join(OUT_DIR, "MERRA2_O3_daily_top5_vs_climatology_30_70hPa_DEBUG.png")
out_pdf = os.path.join(OUT_DIR, "MERRA2_O3_daily_top5_vs_climatology_30_70hPa_DEBUG.pdf")

plt.savefig(out_png, dpi=300)
plt.savefig(out_pdf)
plt.show()

print("\n[INFO] Saved:")
print("  ", out_png)
print("  ", out_pdf)

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# O3 column OCT-SEP evolution comparison
#   - 30–70 hPa
#   - 60–90N polar-cap partial O3 column (DU)
#   - Plot climatology ±1σ
#   - Overlay Top5 lowest-O3 years
#   - Compare:
#       1) MERRA2
#       2) BWCN
#       3) B2000WCN
#       4) B2000WCN_NOCOUPL
#
# Also produce a second figure:
#   - climatology-only comparison in one panel
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

# ============================================================
# Paths
# ============================================================
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
OUT_DIR   = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots/O3_EVOLUTION_COMPARE"
os.makedirs(OUT_DIR, exist_ok=True)

TAG = "30_70hPa"
N_PREV = 91
N_DAYS_EXPECTED = 365

EXTREME_COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

EXP_INFO = {
    "MERRA2": {
        "nc": os.path.join(DATA_BASE, "MERRA2", f"O3_pc_MERRA2_{TAG}.nc"),
        "low10": os.path.join(DATA_BASE, "MERRA2", f"lowest10_years_sorted_{TAG}.txt"),
        "title": "MERRA2",
    },
    "BWCN": {
        "nc": os.path.join(DATA_BASE, "BWCN", f"O3_pc_BWCN_{TAG}.nc"),
        "low10": os.path.join(DATA_BASE, "BWCN", f"lowest10_years_sorted_{TAG}.txt"),
        "title": "BWCN",
    },
    "B2000WCN": {
        "nc": os.path.join(DATA_BASE, "B2000WCN", f"O3_pc_B2000WCN_{TAG}.nc"),
        "low10": os.path.join(DATA_BASE, "B2000WCN", f"lowest10_years_sorted_{TAG}.txt"),
        "title": "B2000WCN",
    },
    "B2000WCN_NOCOUPL": {
        "nc": os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", f"O3_pc_B2000WCN_NOCOUPL_{TAG}.nc"),
        "low10": os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", f"lowest10_years_sorted_{TAG}.txt"),
        "title": "B2000WCN NOCOUPL",
    },
}

# ============================================================
# Style
# ============================================================
rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "axes.linewidth": 0.8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# climatology-only colors
CLIM_COLORS = {
    "MERRA2": "black",
    "BWCN": "#1f77b4",
    "B2000WCN": "#d62728",
    "B2000WCN_NOCOUPL": "#2ca02c",
}

# ============================================================
# Helpers
# ============================================================
def ensure_daily_unique(da):
    da = da.sortby("time")
    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)
    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError("Duplicated daily timestamps detected.")
    return da


def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)


def print_year_counts(da, title="Year counts"):
    years = np.unique(da.time.dt.year.values.astype(int))
    print(f"\n[DEBUG] {title}")
    for y in years:
        cnt = int((da.time.dt.year == y).sum().values)
        print(f"  Year {y}: {cnt} days")


def build_octsep_series(da, year, n_days=365, n_prev=91, debug=False):
    """
    Build one Oct-Sep series for target year:
      Oct-Dec from year-1 + Jan-Sep from year
    """
    cur = da.sel(time=da.time.dt.year == year)
    prev = da.sel(time=da.time.dt.year == (year - 1))

    cur_n = cur.sizes.get("time", 0)
    prev_n = prev.sizes.get("time", 0)

    if debug:
        print(f"[DEBUG] build_octsep_series year={year}")
        print(f"        prev year={year-1}, prev_n={prev_n}")
        print(f"        cur  year={year},   cur_n={cur_n}")

    if prev_n != n_days or cur_n != n_days:
        if debug:
            print(f"        -> INCOMPLETE: expected prev={n_days}, cur={n_days}")
        return None

    cur_arr = np.asarray(cur.values, dtype=float)
    prev_arr = np.asarray(prev.values, dtype=float)

    series = np.concatenate([
        prev_arr[n_days - n_prev:n_days],   # Oct-Dec of previous year
        cur_arr[0:n_days - n_prev]          # Jan-Sep of current year
    ])

    if debug:
        print(f"        -> OK, output length = {len(series)}")

    return series


def build_climatology_octsep(da):
    """
    Build noleap climatology using month-day grouping,
    then reorder to Oct-Sep.
    """
    month = da.time.dt.month.values.astype(int)
    day   = da.time.dt.day.values.astype(int)
    mmdd  = np.array([f"{m:02d}-{d:02d}" for m, d in zip(month, day)])
    da = da.assign_coords(mmdd=("time", mmdd))

    clim_day_mean = da.groupby("mmdd").mean("time")
    clim_day_std  = da.groupby("mmdd").std("time")

    month_days = [
        (10,31), (11,30), (12,31), (1,31), (2,28), (3,31),
        (4,30), (5,31), (6,30), (7,31), (8,31), (9,30)
    ]
    mmdd_order = []
    for m, nd in month_days:
        for d in range(1, nd + 1):
            mmdd_order.append(f"{m:02d}-{d:02d}")

    clim_octsep_mean = clim_day_mean.sel(mmdd=mmdd_order).values
    clim_octsep_std  = clim_day_std.sel(mmdd=mmdd_order).values

    return clim_octsep_mean, clim_octsep_std


def load_low10_years(txt_path):
    if txt_path is None or (not os.path.exists(txt_path)):
        return None
    arr = np.loadtxt(txt_path, dtype=int)
    return np.atleast_1d(arr).reshape(-1)


def build_dataset_bundle(exp_name, debug=False):
    info = EXP_INFO[exp_name]
    nc_path = info["nc"]
    low10_path = info["low10"]

    if not os.path.exists(nc_path):
        raise FileNotFoundError(f"O3 file not found: {nc_path}")
    if not os.path.exists(low10_path):
        raise FileNotFoundError(f"lowest10 txt not found: {low10_path}")

    print(f"\n[INFO] Loading {exp_name}:")
    print(f"       O3  = {nc_path}")
    print(f"       txt = {low10_path}")

    da = xr.load_dataarray(nc_path)
    da = ensure_daily_unique(da)
    da = drop_feb29(da)

    if debug:
        print_year_counts(da, title=f"{exp_name} after drop_feb29")

    clim_mean, clim_std = build_climatology_octsep(da)

    lowest10 = load_low10_years(low10_path)
    top5_years = lowest10[:5]

    print(f"[INFO] {exp_name} lowest10 years: {lowest10}")
    print(f"[INFO] {exp_name} top5 years   : {top5_years}")

    series_top5 = []
    valid_top5_years = []

    for y in top5_years:
        s = build_octsep_series(
            da, int(y),
            n_days=N_DAYS_EXPECTED,
            n_prev=N_PREV,
            debug=debug
        )
        if s is None:
            print(f"[WARN] {exp_name}: skip year {y}, incomplete prev/current year.")
            continue
        series_top5.append(s)
        valid_top5_years.append(int(y))

    series_top5 = np.asarray(series_top5, dtype=float) if len(series_top5) > 0 else np.empty((0, N_DAYS_EXPECTED))

    return {
        "da": da,
        "clim_mean": clim_mean,
        "clim_std": clim_std,
        "top5_years": valid_top5_years,
        "series_top5": series_top5,
        "title": info["title"],
    }


def compute_global_ylim(bundle_dict, pad=3.0):
    vals = []
    for _, b in bundle_dict.items():
        vals.extend(np.ravel(b["clim_mean"] - b["clim_std"]))
        vals.extend(np.ravel(b["clim_mean"] + b["clim_std"]))
        if b["series_top5"].size > 0:
            vals.extend(np.ravel(b["series_top5"]))

    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]

    ymin = np.nanmin(vals)
    ymax = np.nanmax(vals)
    return (ymin - pad, ymax + pad)


def plot_one_panel(ax, bundle, title, ylim):
    x_full = np.arange(N_DAYS_EXPECTED)

    # extreme years
    for i in range(len(bundle["top5_years"])):
        ax.plot(
            x_full,
            bundle["series_top5"][i],
            lw=1.5,
            color=EXTREME_COLORS[i % len(EXTREME_COLORS)],
            label=f"Year {bundle['top5_years'][i]:04d}"
        )

    # climatology envelope
    ax.fill_between(
        x_full,
        bundle["clim_mean"] - bundle["clim_std"],
        bundle["clim_mean"] + bundle["clim_std"],
        color="0.85",
        alpha=0.9,
        linewidth=0,
        zorder=0
    )

    ax.plot(
        x_full,
        bundle["clim_mean"],
        ls="--",
        lw=1.8,
        color="black",
        label=f"{title} climatology"
    )

    ax.set_xlim(0, N_DAYS_EXPECTED)
    ax.set_ylim(ylim)
    ax.set_xticks([0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 334])
    ax.set_xticklabels(["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"])
    ax.set_title(title)
    ax.grid(False)


def add_panel_legend(ax, bundle, key):
    patch_clim = Patch(facecolor="0.85", edgecolor="none", label=f"{bundle['title']} ±1σ")
    line_clim  = Line2D([0], [0], color="black", lw=1.8, ls="--", label=f"{bundle['title']} climatology")

    handles_years = [
        Line2D([0], [0], color=EXTREME_COLORS[i], lw=1.6, label=f"Year {y:04d}")
        for i, y in enumerate(bundle["top5_years"])
    ]

    handles = handles_years + [line_clim, patch_clim]
    ax.legend(handles=handles, loc="best", frameon=False, fontsize=7.5, ncol=2)


def plot_climatology_only(bundles, ylim):
    fig, ax = plt.subplots(1, 1, figsize=(8.0, 4.6), constrained_layout=True)

    x_full = np.arange(N_DAYS_EXPECTED)

    for key in ["MERRA2", "BWCN", "B2000WCN", "B2000WCN_NOCOUPL"]:
        b = bundles[key]
        ax.plot(
            x_full,
            b["clim_mean"],
            lw=2.0,
            color=CLIM_COLORS[key],
            label=b["title"]
        )

    ax.set_xlim(0, N_DAYS_EXPECTED)
    ax.set_ylim(ylim)
    ax.set_xticks([0, 31, 61, 92, 123, 151, 182, 212, 243, 273, 304, 334])
    ax.set_xticklabels(["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"])
    ax.set_ylabel("Partial O$_3$ column, 30–70 hPa (DU)")
    ax.set_xlabel("Month")
    ax.set_title("Climatology only: polar-cap partial ozone column (60–90°N, 30–70 hPa)")
    ax.grid(False)
    ax.legend(loc="best", frameon=False, fontsize=9)

    out_png = os.path.join(OUT_DIR, "O3_climatology_only_compare_MERRA2_BWCN_B2000_NOCOUPL.png")
    out_pdf = os.path.join(OUT_DIR, "O3_climatology_only_compare_MERRA2_BWCN_B2000_NOCOUPL.pdf")

    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

    print("\n[INFO] Saved climatology-only figure:")
    print("  ", out_png)
    print("  ", out_pdf)


# ============================================================
# Main
# ============================================================
if __name__ == "__main__":
    print("=" * 90)
    print("Build O3 evolution comparison figure")
    print("=" * 90)

    bundles = {
        "MERRA2": build_dataset_bundle("MERRA2", debug=False),
        "BWCN": build_dataset_bundle("BWCN", debug=False),
        "B2000WCN": build_dataset_bundle("B2000WCN", debug=False),
        "B2000WCN_NOCOUPL": build_dataset_bundle("B2000WCN_NOCOUPL", debug=False),
    }

    ylim = compute_global_ylim(bundles, pad=3.0)
    print("\n[INFO] Global ylim =", ylim)

    # --------------------------------------------------------
    # Figure 1: 2x2 panel
    # --------------------------------------------------------
    fig, axes = plt.subplots(
        2, 2, figsize=(12.5, 8.0),
        constrained_layout=True,
        sharex=True, sharey=True
    )

    plot_one_panel(axes[0, 0], bundles["MERRA2"], "MERRA2", ylim)
    plot_one_panel(axes[0, 1], bundles["BWCN"], "BWCN", ylim)
    plot_one_panel(axes[1, 0], bundles["B2000WCN"], "B2000WCN", ylim)
    plot_one_panel(axes[1, 1], bundles["B2000WCN_NOCOUPL"], "B2000WCN NOCOUPL", ylim)

    axes[0, 0].set_ylabel("Partial O$_3$ column, 30–70 hPa (DU)")
    axes[1, 0].set_ylabel("Partial O$_3$ column, 30–70 hPa (DU)")
    axes[1, 0].set_xlabel("Month")
    axes[1, 1].set_xlabel("Month")

    add_panel_legend(axes[0, 0], bundles["MERRA2"], "MERRA2")
    add_panel_legend(axes[0, 1], bundles["BWCN"], "BWCN")
    add_panel_legend(axes[1, 0], bundles["B2000WCN"], "B2000WCN")
    add_panel_legend(axes[1, 1], bundles["B2000WCN_NOCOUPL"], "B2000WCN_NOCOUPL")

    fig.suptitle(
        "Polar-cap partial ozone column evolution (60–90°N, 30–70 hPa)",
        fontsize=14,
        y=1.02
    )

    out_png = os.path.join(OUT_DIR, "O3_daily_top5_vs_climatology_compare_MERRA2_BWCN_B2000_NOCOUPL.png")
    out_pdf = os.path.join(OUT_DIR, "O3_daily_top5_vs_climatology_compare_MERRA2_BWCN_B2000_NOCOUPL.pdf")

    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    plt.show()

    print("\n[INFO] Saved 2x2 panel figure:")
    print("  ", out_png)
    print("  ", out_pdf)

    # --------------------------------------------------------
    # Figure 2: climatology only
    # --------------------------------------------------------
    plot_climatology_only(bundles, ylim)

    print("\nDone.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
from datetime import datetime, timedelta

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import pearsonr

# ============================================================
# Paths & Config
# ============================================================
FW_TXT_BWCN    = "/mnt/soclim0/public_data/weiji/BWCN/BWCN_final_warming_date.txt"
FW_TXT_B2000   = "/mnt/soclim0/public_data/weiji/B2000WCN001002/B2000WCN_final_warming_date.txt"
FW_TXT_NOCOUPL = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/B2000WCN_NOCOUPL_final_warming_date.txt"
FW_TXT_MERRA2  = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/MERRA2_final_warming_date.txt"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"
OUT_DIR   = os.path.join(PLOT_BASE, "STATIC_SCATTER_FWD_VS_O3")
os.makedirs(OUT_DIR, exist_ok=True)

BRIDGE_YEAR = 104
APPLY_O3_5D = True

FIG_DPI = 220

BG_COLOR = "0.35"
LOW25_COLOR = "red"
BG_ALPHA = 0.75
LOW25_ALPHA = 0.95
BG_SIZE = 34
LOW25_SIZE = 42

MARK_EXTREMES = True
EXTREME_TOPN = 5
EXTREME_BY = "y_raw"
EXTREME_ASCENDING = True
EXTREME_SIZE = 120
EXTREME_LINEWIDTH = 1.6
EXTREME_COLORS = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']

USE_MANUAL_AXIS_LIMITS = False
MANUAL_XLIM = (60, 150)
MANUAL_YLIM = (60, 170)

# climatology / FWD band styles
CLIM_COLOR = "black"
CLIM_LINESTYLE = "-"
CLIM_LINEWIDTH = 1.6
CLIM_ALPHA = 0.95

FWD_MEAN_COLOR = "#1f77b4"
FWD_BAND_COLOR = "#1f77b4"
FWD_MEAN_LINESTYLE = "--"
FWD_MEAN_LINEWIDTH = 1.2
FWD_BAND_ALPHA = 0.10


# ============================================================
# Helpers
# ============================================================
def doy_to_mmdd(x, pos):
    if np.isnan(x) or x < 1 or x > 365:
        return ""
    base_date = datetime(2001, 1, 1)
    target_date = base_date + timedelta(days=int(round(x)) - 1)
    return target_date.strftime("%m-%d")


def load_fwd_data(txt_path):
    fwd_dict = {}
    if not os.path.exists(txt_path):
        print(f"Warning: FWD file not found -> {txt_path}")
        return fwd_dict

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("#") or not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                year = int(parts[0])
                doy_str = parts[1]
                if doy_str.lower() != "nan":
                    fwd_dict[year] = float(doy_str)
    return fwd_dict


def assert_daily_unique(da, name=""):
    da = da.sortby("time")
    yy = da.time.dt.year.values.astype(int)
    mm = da.time.dt.month.values.astype(int)
    dd = da.time.dt.day.values.astype(int)
    key = yy * 10000 + mm * 100 + dd
    if np.unique(key).size != key.size:
        raise ValueError(f"{name}: duplicated daily timestamps detected.")
    return da


def drop_feb29(da):
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)


def build_mmdd_order():
    month_days = [
        (1, 31), (2, 28), (3, 31), (4, 30), (5, 31), (6, 30),
        (7, 31), (8, 31), (9, 30), (10, 31), (11, 30), (12, 31)
    ]
    out = []
    for m, nd in month_days:
        for d in range(1, nd + 1):
            out.append(f"{m:02d}-{d:02d}")
    return out


MMDD_ORDER = build_mmdd_order()
MMDD_TO_DOY = {mmdd: i + 1 for i, mmdd in enumerate(MMDD_ORDER)}


def get_ma_min_o3_series(o3_nc, apply_o3_5d=True):
    if not os.path.exists(o3_nc):
        print(f"Warning: O3 file not found -> {o3_nc}")
        return {}

    da = xr.open_dataarray(o3_nc)
    da = assert_daily_unique(da, name="O3").sortby("time")

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    years = np.unique(da.time.dt.year.values.astype(int))
    o3_dict = {}

    for yr in years:
        mask = (da.time.dt.year == yr) & (da.time.dt.month.isin([3, 4]))
        seg = da.where(mask, drop=True)
        n_valid = int(seg.count().values) if seg.size > 0 else 0
        if n_valid >= 58:
            o3_dict[int(yr)] = float(seg.min().values)

    da.close()
    return o3_dict


def get_low25_years(txt_path):
    if txt_path and os.path.exists(txt_path):
        return set(np.loadtxt(txt_path, dtype=int).reshape(-1))
    return set()


def load_o3_daily_climatology_curve(o3_nc, apply_o3_5d=True, drop_years=None):
    """
    返回 climatology curve:
      x_clim: 1..365 (DOY, noleap)
      y_clim: daily climatological partial O3 column
    """
    if not os.path.exists(o3_nc):
        print(f"Warning: O3 file not found -> {o3_nc}")
        return None

    da = xr.open_dataarray(o3_nc)
    da = assert_daily_unique(da, name="O3_daily").sortby("time")
    da = drop_feb29(da)

    if drop_years:
        da = da.sel(time=~da.time.dt.year.isin(list(drop_years)))

    if apply_o3_5d:
        da = da.rolling(time=5, center=True, min_periods=5).mean()

    mmdd = np.array([f"{m:02d}-{d:02d}" for m, d in zip(
        da.time.dt.month.values.astype(int),
        da.time.dt.day.values.astype(int)
    )])
    da = da.assign_coords(mmdd=("time", mmdd))

    clim = da.groupby("mmdd").mean("time")
    clim = clim.sel(mmdd=MMDD_ORDER)

    x_clim = np.arange(1, 366, dtype=float)
    y_clim = np.asarray(clim.values, dtype=float)

    da.close()
    return {"x": x_clim, "y": y_clim}


def average_climatology_curves(curves):
    curves = [c for c in curves if c is not None]
    if len(curves) == 0:
        return None
    if len(curves) == 1:
        return curves[0]

    x = curves[0]["x"]
    y_stack = np.vstack([c["y"] for c in curves])
    return {"x": x, "y": np.nanmean(y_stack, axis=0)}


def build_scatter_df(fwd_txt, o3_nc, low25_txt, part_name, is_bridge_case=False):
    fwd_data = load_fwd_data(fwd_txt)
    o3_data = get_ma_min_o3_series(o3_nc, apply_o3_5d=APPLY_O3_5D)
    low25_set = get_low25_years(low25_txt)

    common_years = sorted(list(set(fwd_data.keys()).intersection(set(o3_data.keys()))))

    if is_bridge_case and BRIDGE_YEAR in common_years:
        common_years.remove(BRIDGE_YEAR)

    x_list, y_list, low25_mask, year_list, part_list = [], [], [], [], []
    for y in common_years:
        x_list.append(fwd_data[y])
        y_list.append(o3_data[y])
        low25_mask.append(y in low25_set)
        year_list.append(y)
        part_list.append(part_name)

    return {
        "x": np.array(x_list, dtype=float),
        "y": np.array(y_list, dtype=float),
        "y_raw": np.array(y_list, dtype=float),
        "year": np.array(year_list, dtype=int),
        "part": np.array(part_list, dtype=object),
        "is_low25": np.array(low25_mask, dtype=bool)
    }


def combine_dfs(df1, df2):
    if df1 is None or df1["x"].size == 0:
        return df2
    if df2 is None or df2["x"].size == 0:
        return df1
    return {
        "x": np.concatenate([df1["x"], df2["x"]]),
        "y": np.concatenate([df1["y"], df2["y"]]),
        "y_raw": np.concatenate([df1["y_raw"], df2["y_raw"]]),
        "year": np.concatenate([df1["year"], df2["year"]]),
        "part": np.concatenate([df1["part"], df2["part"]]),
        "is_low25": np.concatenate([df1["is_low25"], df2["is_low25"]])
    }


def select_panel_extremes(df, topn=5, extreme_by="y_raw", ascending=True):
    if df is None or len(df["year"]) == 0:
        return np.array([], dtype=int)

    vals = np.asarray(df[extreme_by], dtype=float)
    valid_idx = np.where(np.isfinite(vals))[0]
    if valid_idx.size == 0:
        return np.array([], dtype=int)

    order = np.argsort(vals[valid_idx])
    if not ascending:
        order = order[::-1]

    keep = valid_idx[order[:min(topn, len(order))]]
    return keep


def overlay_panel_extremes(ax, df, idx_extreme):
    if df is None or len(idx_extreme) == 0:
        return []

    legend_entries = []

    for i, idx in enumerate(idx_extreme):
        c = EXTREME_COLORS[i % len(EXTREME_COLORS)]
        x = float(df["x"][idx])
        y = float(df["y"][idx])
        part = str(df["part"][idx])
        year = int(df["year"][idx])

        if part == "BWCN":
            label = f"BWCN-{year:04d}"
        else:
            label = f"{year:04d}"

        legend_entries.append((i, label, c, part))

        if part == "BWCN":
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                facecolors="none",
                edgecolors=c,
                linewidths=EXTREME_LINEWIDTH,
                zorder=10
            )
        else:
            ax.scatter(
                x, y,
                s=EXTREME_SIZE,
                color=c,
                edgecolors="k",
                linewidths=0.8,
                zorder=10
            )

    return legend_entries


def compute_global_axis_limits(dfs, clim_curves=None, use_manual=False,
                               manual_xlim=None, manual_ylim=None,
                               xpad=5.0, ypad=3.0):
    if use_manual:
        return manual_xlim, manual_ylim

    all_x = []
    all_y = []

    for df in dfs:
        if df is not None and df["x"].size > 0:
            all_x.extend(df["x"])
            all_y.extend(df["y"])

    if clim_curves is not None:
        for c in clim_curves:
            if c is not None:
                all_x.extend(c["x"])
                all_y.extend(c["y"])

    all_x = np.asarray(all_x, dtype=float)
    all_y = np.asarray(all_y, dtype=float)

    if all_x.size == 0 or all_y.size == 0:
        return (60, 150), (60, 170)

    xmin, xmax = np.nanmin(all_x), np.nanmax(all_x)
    ymin, ymax = np.nanmin(all_y), np.nanmax(all_y)

    return (xmin - xpad, xmax + xpad), (ymin - ypad, ymax + ypad)


# ============================================================
# Plotting
# ============================================================
def draw_one_panel(ax, df, clim_curve, panel_title, xlim, ylim):
    ax.grid(True, linestyle=":", alpha=0.45)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # 右侧 Y 轴用于 climatology curve
    ax2 = ax.twinx()
    ax2.set_ylim(ylim)
    ax2.set_ylabel(r"O$_3$ climatology (DU)", color=CLIM_COLOR)
    ax2.tick_params(axis='y', colors=CLIM_COLOR)

    if df is None or len(df["x"]) == 0:
        ax.set_title(panel_title, fontweight="bold")
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel("Final Warming Date")
        ax.set_ylabel(r"MA min O$_3$ (DU)")
        ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))
        return

    low_mask = df["is_low25"]
    bg_mask = ~low_mask

    # scatter points
    if np.any(bg_mask):
        ax.scatter(
            df["x"][bg_mask], df["y"][bg_mask],
            s=BG_SIZE, color=BG_COLOR, alpha=BG_ALPHA,
            edgecolors="none", zorder=2
        )

    if np.any(low_mask):
        ax.scatter(
            df["x"][low_mask], df["y"][low_mask],
            s=LOW25_SIZE, color=LOW25_COLOR, alpha=LOW25_ALPHA,
            edgecolors="k", linewidths=0.4, zorder=5
        )

    # extreme points
    legend_entries = []
    if MARK_EXTREMES:
        idx_ext = select_panel_extremes(
            df,
            topn=EXTREME_TOPN,
            extreme_by=EXTREME_BY,
            ascending=EXTREME_ASCENDING
        )
        legend_entries = overlay_panel_extremes(ax, df, idx_ext)

    # FWD mean ± sigma band + mean line
    mean_fwd = np.nanmean(df["x"])
    std_fwd  = np.nanstd(df["x"], ddof=0)
    

    # mean ± 1σ band
    ax.axvspan(mean_fwd - std_fwd, mean_fwd + std_fwd,
            color=FWD_BAND_COLOR, alpha=0.18, zorder=0)

    # mean line
    ax.axvline(mean_fwd, color=FWD_MEAN_COLOR, linestyle="-",
            linewidth=1.4, alpha=0.95, zorder=1)

    # ±1σ boundary lines
    ax.axvline(mean_fwd - std_fwd, color=FWD_MEAN_COLOR, linestyle="--",
            linewidth=1.0, alpha=0.9, zorder=1)
    ax.axvline(mean_fwd + std_fwd, color=FWD_MEAN_COLOR, linestyle="--",
            linewidth=1.0, alpha=0.9, zorder=1)

    # climatology curve on right y-axis
    if clim_curve is not None:
        ax2.plot(
            clim_curve["x"], clim_curve["y"],
            color=CLIM_COLOR, linestyle=CLIM_LINESTYLE,
            linewidth=CLIM_LINEWIDTH, alpha=CLIM_ALPHA, zorder=1
        )

    # statistics
    n_all = len(df["x"])
    if n_all >= 3:
        try:
            r_all, p_all = pearsonr(df["x"], df["y"])
            txt_all = f"ALL\nr = {r_all:.3f}\np = {p_all:.2e}\nN = {n_all}"
        except Exception:
            txt_all = f"ALL\nN = {n_all}"
    else:
        txt_all = f"ALL\nN = {n_all}"

    ax.text(
        0.04, 0.96, txt_all,
        transform=ax.transAxes, va="top", ha="left", fontsize=9.2, color="k",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor="0.7")
    )

    n_low = int(np.sum(low_mask))
    if n_low >= 3:
        try:
            r_low, p_low = pearsonr(df["x"][low_mask], df["y"][low_mask])
            txt_low = f"LOW25\nr = {r_low:.3f}\np = {p_low:.2e}\nN = {n_low}"
        except Exception:
            txt_low = f"LOW25\nN = {n_low}"
    else:
        txt_low = f"LOW25\nN = {n_low}"

    ax.text(
        0.96, 0.96, txt_low,
        transform=ax.transAxes, va="top", ha="right", fontsize=9.2, color=LOW25_COLOR,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.35, edgecolor=LOW25_COLOR)
    )

    # legend
    handles = []
    labels = []

    for i, label, c, part in legend_entries:
        if part == "BWCN":
            h = Line2D(
                [0], [0],
                marker="o",
                color="none",
                markeredgecolor=c,
                markerfacecolor="none",
                markeredgewidth=1.2,
                markersize=5.0,
                linewidth=0
            )
        else:
            h = Line2D(
                [0], [0],
                marker="o",
                color="none",
                markeredgecolor="k",
                markerfacecolor=c,
                markeredgewidth=0.6,
                markersize=5.0,
                linewidth=0
            )
        handles.append(h)
        labels.append(f"Top{i+1}: {label}")

    handles.extend([
        Line2D([0], [0], color=CLIM_COLOR, lw=CLIM_LINEWIDTH, ls=CLIM_LINESTYLE),
        Line2D([0], [0], color=FWD_MEAN_COLOR, lw=FWD_MEAN_LINEWIDTH, ls=FWD_MEAN_LINESTYLE),
        Patch(facecolor=FWD_BAND_COLOR, edgecolor="none", alpha=FWD_BAND_ALPHA),
    ])
    labels.extend([
        "O$_3$ climatology",
        f"Mean FWD: {doy_to_mmdd(mean_fwd, None)}",
        r"FWD $\pm 1\sigma$"
    ])

    ax.legend(
        handles, labels,
        loc="lower right",
        fontsize=4.8,
        frameon=False,
        handletextpad=0.35,
        borderpad=0.2,
        labelspacing=0.25
    )

    ax.set_title(panel_title, fontweight="bold")
    ax.set_xlabel("Final Warming Date")
    ax.set_ylabel(r"MA min O$_3$ (DU)")
    ax.xaxis.set_major_formatter(FuncFormatter(doy_to_mmdd))


# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    print("Loading data...")

    # scatter data
    df_merra = build_scatter_df(
        FW_TXT_MERRA2,
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        os.path.join(DATA_BASE, "MERRA2", "low25pct_years_30_70hPa.txt"),
        part_name="MERRA2",
        is_bridge_case=False
    )

    df_bwcn = build_scatter_df(
        FW_TXT_BWCN,
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "BWCN", "low25pct_years_30_70hPa.txt"),
        part_name="BWCN",
        is_bridge_case=False
    )

    df_b2000 = build_scatter_df(
        FW_TXT_B2000,
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN", "low25pct_years_30_70hPa.txt"),
        part_name="B2000WCN",
        is_bridge_case=True
    )

    df_coupled = combine_dfs(df_bwcn, df_b2000)

    df_nocoupl = build_scatter_df(
        FW_TXT_NOCOUPL,
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "low25pct_years_30_70hPa.txt"),
        part_name="B2000WCN_NOCOUPL",
        is_bridge_case=True
    )

    # climatology curves
    clim_merra = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "MERRA2", "O3_pc_MERRA2_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years=None
    )

    clim_bwcn = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "BWCN", "O3_pc_BWCN_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years=None
    )

    clim_b2000 = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "B2000WCN", "O3_pc_B2000WCN_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years={BRIDGE_YEAR}
    )

    clim_coupled = average_climatology_curves([clim_bwcn, clim_b2000])

    clim_nocoupl = load_o3_daily_climatology_curve(
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL", "O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        apply_o3_5d=APPLY_O3_5D,
        drop_years={BRIDGE_YEAR}
    )

    # global axes
    xlim, ylim = compute_global_axis_limits(
        [df_merra, df_coupled, df_nocoupl],
        clim_curves=None,
        use_manual=USE_MANUAL_AXIS_LIMITS,
        manual_xlim=MANUAL_XLIM,
        manual_ylim=MANUAL_YLIM
    )

    print("Global xlim =", xlim)
    print("Global ylim =", ylim)

    print("Plotting...")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5.8), dpi=FIG_DPI)
    plt.subplots_adjust(wspace=0.30)

    draw_one_panel(axes[0], df_merra,   clim_merra,   "MERRA2",             xlim, ylim)
    draw_one_panel(axes[1], df_coupled, clim_coupled, "B2000WCN (Coupled)", xlim, ylim)
    draw_one_panel(axes[2], df_nocoupl, clim_nocoupl, "B2000WCN NOCOUPL",   xlim, ylim)

    fig.suptitle("Scatter: MA min O$_3$ vs Final Warming Date (FWD)", fontsize=16, fontweight="bold", y=1.03)

    for ax in axes:
        for tick in ax.get_xticklabels():
            tick.set_rotation(45)

    save_path = os.path.join(OUT_DIR, "Scatter_FWD_vs_O3_MERRA2_B2000_NOCOUPL_extreme_climcurve_fwdband.png")
    plt.savefig(save_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

    print(f"Plot saved successfully at:\n{save_path}")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

# ============================================================
# 1. 路径与配置
# ============================================================
DATA_NC = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data/MERRA2/O3_pc_MERRA2_30_70hPa.nc"
OUT_DIR = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots/O3_MIN_COMPARE"
os.makedirs(OUT_DIR, exist_ok=True)

# 设置你要分组的年份数量 N
N_YEARS = 5

# 绘图样式
rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "DejaVu Sans"],
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ============================================================
# 2. 数据处理与辅助函数
# ============================================================
def drop_feb29(da):
    """剔除闰年 2月29日"""
    mask = ~((da.time.dt.month == 2) & (da.time.dt.day == 29))
    return da.sel(time=mask)

def get_mmdd(da):
    """生成 mm-dd 字符串数组用于对齐气候态"""
    m = da.time.dt.month.values
    d = da.time.dt.day.values
    return np.array([f"{mm:02d}-{dd:02d}" for mm, dd in zip(m, d)])

def doy_to_date_str(doy):
    """将非闰年的 DOY 转换为易读的日期字符串 (如 Mar-15)"""
    dt = pd.to_datetime(f"2023-{int(doy)}", format="%Y-%j")
    return dt.strftime("%b-%d")

# ============================================================
# 3. 主程序
# ============================================================
print("[INFO] 加载 MERRA2 臭氧数据...")
da = xr.load_dataarray(DATA_NC)
da = drop_feb29(da)

# 统一附上 mmdd 坐标
mmdd_coords = get_mmdd(da)
da = da.assign_coords(mmdd=("time", mmdd_coords))

# 计算气候态 (Mean & Std)
clim_mean = da.groupby("mmdd").mean("time")
clim_std = da.groupby("mmdd").std("time")

# 计算距平 (Anomaly) = Daily - Climatology
da_anom = da.groupby("mmdd") - clim_mean

# 计算 5 days 滑动平均 (MA)
da_5d = da.rolling(time=5, center=True, min_periods=5).mean()
da_anom_5d = da_anom.rolling(time=5, center=True, min_periods=5).mean()

# 提取 3月 - 4月 窗口 (Mar=3, Apr=4)
spring_mask = da.time.dt.month.isin([3, 4])
da_5d_sp = da_5d.sel(time=spring_mask)
da_anom_5d_sp = da_anom_5d.sel(time=spring_mask)

# 提取有效年份并计算两种 minimum
years = np.unique(da_5d_sp.time.dt.year.values)
results = []

print("[INFO] 计算每年 3-4 月的两种 Minimums...")
for y in years:
    sub_abs = da_5d_sp.sel(time=da_5d_sp.time.dt.year == y)
    sub_ano = da_anom_5d_sp.sel(time=da_anom_5d_sp.time.dt.year == y)
    
    # 过滤不完整的年份（3、4月共 61 天）
    if len(sub_abs) < 61:
        continue
        
    # --- 方法A: 绝对值最低 ---
    idx_min_abs = sub_abs.argmin(dim="time").values
    abs_min_val = sub_abs[idx_min_abs].values
    abs_min_date = sub_abs.time[idx_min_abs].values
    
    # --- 方法B: 偏离气候态最低 (距平最低) ---
    idx_min_ano = sub_ano.argmin(dim="time").values
    ano_min_absval = sub_abs[idx_min_ano].values # 提取发生异常最低这天的绝对O3值
    ano_min_date = sub_ano.time[idx_min_ano].values
    
    results.append({
        "Year": y,
        "Abs_Min_Val": float(abs_min_val),
        "Abs_Min_DOY": pd.to_datetime(abs_min_date).dayofyear,
        "Ano_Min_AbsVal": float(ano_min_absval),
        "Ano_Min_DOY": pd.to_datetime(ano_min_date).dayofyear
    })

df = pd.DataFrame(results)

# 全局平均极值和平均日期
mean_abs_min_val = df["Abs_Min_Val"].mean()
mean_abs_min_doy = df["Abs_Min_DOY"].mean()
mean_ano_min_val = df["Ano_Min_AbsVal"].mean()
mean_ano_min_doy = df["Ano_Min_DOY"].mean()

# ============================================================
# 按绝对值最低 (Abs_Min_Val) 排序并动态分组
# ============================================================
df_sorted = df.sort_values(by="Abs_Min_Val").reset_index(drop=True)
total_years = len(df_sorted)

# 安全检查：确保 N_YEARS 不会超过总年份数的三分之一，防止切片重叠
if N_YEARS * 3 > total_years:
    print(f"[WARN] 设定的 N_YEARS ({N_YEARS}) 过大，总有效年份仅为 {total_years}。存在重叠风险。")

group_min_N = df_sorted.head(N_YEARS)
group_max_N = df_sorted.tail(N_YEARS)

# Middle N 逻辑：自动适配奇偶数
mid_idx = total_years // 2
half_n = N_YEARS // 2
remainder = N_YEARS % 2
group_mid_N = df_sorted.iloc[mid_idx - half_n : mid_idx + half_n + remainder]

groups = {
    f"Lowest_{N_YEARS}_Years": group_min_N,
    f"Middle_{N_YEARS}_Years": group_mid_N,
    f"Highest_{N_YEARS}_Years": group_max_N
}

# ============================================================
# 4. 绘图模块
# ============================================================
# 构建 3-4 月的 X 轴 (Day of Year 60 到 120, 平年)
doy_spring = np.arange(60, 121) 
mmdd_spring = [pd.to_datetime(f"2023-{d}", format="%Y-%j").strftime("%m-%d") for d in doy_spring]

clim_mean_sp = clim_mean.sel(mmdd=mmdd_spring).values
clim_std_sp = clim_std.sel(mmdd=mmdd_spring).values

# 动态生成颜色：循环使用 tab10 色谱，支持任意 N
colors = [plt.cm.tab10.colors[i % 10] for i in range(N_YEARS)]

print(f"[INFO] 开始绘制分组对比图 (N={N_YEARS})...")
for group_name, df_grp in groups.items():
    fig, ax = plt.subplots(figsize=(9, 5.5), constrained_layout=True)
    
    # 1. 绘制气候态背景及 ±1σ 包络区 
    ax.fill_between(doy_spring, clim_mean_sp - clim_std_sp, clim_mean_sp + clim_std_sp, 
                    color="gray", alpha=0.2, lw=0, label="Climatology ±1σ")
    ax.plot(doy_spring, clim_mean_sp, color="gray", lw=2, ls="--", label="Climatology Mean")
    
    # 2. 绘制气候态全局的平均极值点与标注
    # 绝对法平均
    ax.axhline(mean_abs_min_val, color="black", ls="-", lw=1, alpha=0.6)
    ax.axvline(mean_abs_min_doy, color="black", ls="-", lw=1, alpha=0.6)
    ax.text(mean_abs_min_doy + 1, mean_abs_min_val + 1, 
            f"Abs Mean:\n{mean_abs_min_val:.1f} DU\n{doy_to_date_str(mean_abs_min_doy)}", 
            color="black", fontsize=9, va="bottom")
            
    # 异常法平均
    ax.axhline(mean_ano_min_val, color="blue", ls="--", lw=1, alpha=0.6)
    ax.axvline(mean_ano_min_doy, color="blue", ls="--", lw=1, alpha=0.6)
    ax.text(mean_ano_min_doy - 1, mean_ano_min_val - 1, 
            f"Ano Mean:\n{mean_ano_min_val:.1f} DU\n{doy_to_date_str(mean_ano_min_doy)}", 
            color="blue", fontsize=9, ha="right", va="top")
    
    # 3. 绘制组内各年份的日变化曲线及极值点
    for i, row in enumerate(df_grp.itertuples()):
        y = int(row.Year)
        sub_abs = da_5d_sp.sel(time=da_5d_sp.time.dt.year == y).values
        
        ax.plot(doy_spring, sub_abs, lw=1.2, color=colors[i], alpha=0.8, label=f"Year {y}")
        
        # 标出 绝对最低值 (实心圆)
        ax.scatter(row.Abs_Min_DOY, row.Abs_Min_Val, 
                   color=colors[i], marker="o", s=60, edgecolors='black', zorder=5)
        
        # 标出 异常最低值 (五角星)
        ax.scatter(row.Ano_Min_DOY, row.Ano_Min_AbsVal, 
                   color=colors[i], marker="*", s=120, edgecolors='black', zorder=5)

    # 美化坐标轴
    ax.set_xlim(60, 120)
    ax.set_ylim(75, 130)  # 强制固定 Y 轴
    
    xticks_doy = [60, 75, 91, 106, 120]
    xticks_labels = ["Mar 1", "Mar 16", "Apr 1", "Apr 16", "Apr 30"]
    ax.set_xticks(xticks_doy)
    ax.set_xticklabels(xticks_labels)
    
    ax.set_ylabel("Partial O$_3$ column (30–70 hPa) [DU]")
    ax.set_title(f"MERRA2 Mar-Apr O$_3$ Minimum Test - {group_name.replace('_', ' ')}")
    
    # 图例设置
    handles, labels = ax.get_legend_handles_labels()
    patch_clim = Patch(facecolor="gray", alpha=0.2, edgecolor="none", label="Climatology ±1σ")
    
    new_handles = []
    new_labels = []
    for h, l in zip(handles, labels):
        if l not in ["Climatology ±1σ"]:
            new_handles.append(h)
            new_labels.append(l)
    
    new_handles.insert(0, patch_clim)
    new_labels.insert(0, "Climatology ±1σ")
            
    new_handles.append(Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markeredgecolor='black', markersize=8))
    new_labels.append('Method A: Abs Minimum')
    new_handles.append(Line2D([0], [0], marker='*', color='w', markerfacecolor='gray', markeredgecolor='black', markersize=12))
    new_labels.append('Method B: Max Neg Anomaly')
    
    ax.legend(new_handles, new_labels, loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9)
    
    out_file = os.path.join(OUT_DIR, f"{group_name}_O3_Compare_FixedY.png")
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"  已保存: {out_file}")

print("[INFO] 全部完成！")